# B站爬虫技术设计

> 基于 `bilibili-api-python` 库（[GitHub: Nemo2011/bilibili-api](https://github.com/Nemo2011/bilibili-api)），已对照 [PyPI 官方文档](https://pypi.org/project/bilibili-api-python/) 确认 API 返回格式。

## 环境安装

```bash
pip3 install bilibili-api-python        # 主库
pip3 install aiohttp                    # 异步请求库（也可选 httpx / curl_cffi）
```

> 模块按照 `curl_cffi` → `aiohttp` → `httpx` 的优先级选择请求库。若需更高的反爬能力，可安装 `curl_cffi` 并设置浏览器指纹伪装：
> ```python
> from bilibili_api import request_settings
> request_settings.set("impersonate", "chrome131")
> ```

## 使用的两个 API 方法

### `search.search_by_type()` — 搜索视频
```python
from bilibili_api import search
from bilibili_api.search import SearchObjectType, OrderVideo

result = await search.search_by_type(
    keyword="IA",
    search_type=SearchObjectType.VIDEO,
    order_type=OrderVideo.CLICK,  # CLICK=播放量 / PUBDATE=发布时间
    page=1
)
# result = {"numResults": ..., "numPages": ..., "result": [...]}
```

### `video.Video(bvid).get_info()` — 获取视频详情
```python
from bilibili_api import video

v = video.Video(bvid="BV1uv411q7Mv")
info = await v.get_info()
```

**无需登录（Credential）**即可调用以上两个方法，仅当需要点赞/评论等写操作时才需要登录态。

### 为什么需要两个方法
`search_by_type()` 返回大部分基础字段，但缺少 `coin`, `share`, `like`, `tname`, `copyright`。这些必须从 `get_info()` 获取。搜索约 1次/50条，详情每视频 1次——后者是爬取时间的主要开销。

## `get_info()` 返回格式（源自 PyPI 官方文档）

```python
{
    "bvid":      "BV1uv411q7Mv",
    "aid":       243922477,
    "videos":    1,                  # 分P数
    "tid":       17,                 # 分区ID
    "tname":     "单机游戏",          # 分区名称 → CSV: category
    "copyright": 1,                  # 1=自制, 2=转载 → 辅助判断 is_reupload
    "pic":       "https://...",      # 封面图
    "title":     "爆肝９８小时！...",   # 标题（干净，无HTML标签）
    "pubdate":   1595203214,         # Unix时间戳(int)
    "ctime":     1595168654,         # 创建时间戳
    "duration":  270,                # 时长，秒(int)
    "owner": {
        "mid":  12345678,           # UP主UID
        "name": "某UP主",            # UP主昵称
        "face": "https://..."       # UP主头像
    },
    "stat": {
        "view":     500000,         # 播放量
        "danmaku":  3000,           # 弹幕数
        "reply":    800,            # 评论数
        "favorite": 5000,           # 收藏数
        "coin":     1200,           # 硬币数
        "share":    300,            # 分享数
        "like":     8500            # 点赞数
    },
    ...  # 更多字段省略
}
```

> **来源**：PyPI 页面 `video.Video(bvid="BV1uv411q7Mv").get_info()` 的示例输出（已格式化并补充 `stat` 和 `owner` 结构）。

## API返回字段 → CSV目标字段 映射

| # | CSV列名 | 来源 | 库方法 / 字段路径 | 转换说明 |
|---|---------|------|-------------------|---------|
| 1 | `aid` | 搜索 | `item["aid"]` | 直接 |
| 2 | `bvid` | 搜索 | `item["bvid"]` | 复合去重键之一 |
| 3 | `page` | 详情 | `p["page"]` | 分P页码；单P=1，多P为实际页码 |
| 4 | `mid` | 搜索 | `item["mid"]` | 直接，UP主唯一UID |
| 5 | `title` | 搜索 | `item["title"]` | **需清洗**：`re.sub(r'<[^>]+>', '', title)` 移除 `<em>` 标签 |
| 6 | `author` | 搜索 | `item["author"]` | 直接 |
| 7 | `publish_date` | 搜索 | `item["pubdate"]` | **需转换**：`datetime.fromtimestamp(pubdate)` |
| 8 | `duration_sec` | 详情 | `info["duration"]` | 详情返回 int 秒数；搜索返回 `"MM:SS"`，不用 |
| 9 | `play_count` | 搜索 | `item["play"]` | 多P时按P数均分 |
| 10 | `danmaku_count` | 搜索 | `item["video_review"]` | 直接 |
| 11 | `comment_count` | 搜索 | `item["review"]` | 直接 |
| 12 | `favorite_count` | 搜索 | `item["favorites"]` | 直接 |
| 13 | `coin_count` | **详情** | `info["stat"]["coin"]` | 搜索无此字段 |
| 14 | `share_count` | **详情** | `info["stat"]["share"]` | 搜索无此字段 |
| 15 | `like_count` | **详情** | `info["stat"]["like"]` | 搜索无此字段 |
| 16 | `tags` | 搜索 | `item["tag"]` | 直接（逗号分隔） |
| 17 | `category` | 搜索 | `item["typename"]` | 搜索API自带分区名，比详情API的 `tname` 更可靠（后者常为空） |
| 18 | `url` | 构造 | - | `f"https://www.bilibili.com/video/{bvid}"` |
| 19 | `copyright` | 详情 | `info["copyright"]` | 1=自制, 2=转载 |
| 20 | `play_estimated` | 构造 | - | 多P均分时为 True，单P为 False |

### 实际产出

- 爬取时间：~60 分钟（搜索 4 分钟 + 详情 50 分钟）
- 数据量：**4,128 行**（去重后），20 列
- 单P：~1,890 条；多P展开：~2,240 条

## 搜索返回值详细格式

`search_by_type()` 返回的 `result[]` 中每条视频条目：

| 字段 | 类型 | 示例值 | 映射到CSV |
|------|------|--------|----------|
| `aid` | int | `114514` | `aid` |
| `bvid` | str | `"BV1xx411c7mD"` | `bvid` |
| `mid` | int | `123456` | `mid` |
| `title` | str | `"...<em class=\"keyword\">IA</em>..."` | `title`（去标签后） |
| `author` | str | `"某UP主"` | `author` |
| `play` | int | `500000` | `play_count` |
| `duration` | str | `"4:30"` | 不用（用详情接口的 int 秒数） |
| `pubdate` | int | `1700000000` | `publish_date`→datetime |
| `video_review` | int | `3000` | `danmaku_count` |
| `review` | int | `800` | `comment_count` |
| `favorites` | int | `5000` | `favorite_count` |
| `tag` | str | `"VOCALOID,IA,原创曲"` | `tags` |
| `typeid` | int | `30` | 参考用 |

> `title` 含 `<em class="keyword">` HTML标签，须 `re.sub(r'<[^>]+>', '', title)` 清除。`duration` 为 `"MM:SS"` 字符串，改用详情接口的 `int`。`mid` 免费附带，已纳入CSV。

## 详情返回值（从 `get_info()` 获取的字段）

| CSV目标 | 提取路径 | 搜索已有？ |
|---------|---------|:---:|
| `duration_sec` | `info["duration"]` | 否（搜索是字符串） |
| `coin_count` | `info["stat"]["coin"]` | 否 |
| `share_count` | `info["stat"]["share"]` | 否 |
| `like_count` | `info["stat"]["like"]` | 否 |
| `category` | `info["tname"]` | 否 |
| `copyright` (辅助) | `info["copyright"]` | 否，1=自制, 2=转载 |

## 爬虫整体流程

### 搜索配置

按分区严格程度分成两档，先严格后放宽：

```
第1档：分区限定 VOCALOID·UTAU (tid=30)
  关键词 × 排序:
    "IA"              × CLICK, PUBDATE
    "IA オリジナル曲"  × CLICK, PUBDATE
    "IA 翻唱"          × CLICK, PUBDATE
  → 6 组搜索配置

第2档：不限分区（仅在总数不足1200时启用）
  关键词 × 排序:
    "IA"              × CLICK, PUBDATE
  → 2 组搜索配置（其他杂项分区也会被收录）
```

### 阶段1：批量搜索

```python
import asyncio
from bilibili_api import search
from bilibili_api.search import SearchObjectType, OrderVideo
from datetime import datetime
import re

known_bvids = load_existing_bvids("data/ia_music_data.csv")
new_videos = []

# 搜索配置：(关键词, 排序, 分区tid, 配置描述)
SEARCH_CONFIGS = [
    # === 第1档：限定 VOCALOID·UTAU 分区 ===
    ("IA",              OrderVideo.CLICK,   30, "IA/播放量/VOCALOID"),
    ("IA",              OrderVideo.PUBDATE, 30, "IA/最新/VOCALOID"),
    ("IA オリジナル曲",  OrderVideo.CLICK,   30, "IA原创曲/播放量/VOCALOID"),
    ("IA オリジナル曲",  OrderVideo.PUBDATE, 30, "IA原创曲/最新/VOCALOID"),
    ("IA 翻唱",          OrderVideo.CLICK,   30, "IA翻唱/播放量/VOCALOID"),
    ("IA 翻唱",          OrderVideo.PUBDATE, 30, "IA翻唱/最新/VOCALOID"),
]

# 第2档：不限分区（仅在总数不足时使用）
FALLBACK_CONFIGS = [
    ("IA", OrderVideo.CLICK,   None, "IA/播放量/全部分区"),
    ("IA", OrderVideo.PUBDATE, None, "IA/最新/全部分区"),
]

async def search_phase():
    for keyword, order, zone_id, label in SEARCH_CONFIGS:
        page = 1
        print(f"搜索: {label}")
        while True:
            await random_delay()
            kwargs = {
                "keyword": keyword,
                "search_type": SearchObjectType.VIDEO,
                "order_type": order,
                "page": page,
            }
            if zone_id is not None:
                kwargs["video_zone_type"] = zone_id

            result = await search.search_by_type(**kwargs)
            items = result.get("result", [])
            if not items:
                print(f"  第{page}页无结果，{label}搜索结束")
                break

            for item in items:
                bvid = item["bvid"]
                if bvid in known_bvids:
                    continue
                new_videos.append({
                    "aid":            item["aid"],
                    "bvid":           bvid,
                    "mid":            item["mid"],
                    "title":          re.sub(r'<[^>]+>', '', item["title"]),
                    "author":         item["author"],
                    "publish_date":   datetime.fromtimestamp(item["pubdate"]),
                    "play_count":     item["play"],
                    "danmaku_count":  item["video_review"],
                    "comment_count":  item["review"],
                    "favorite_count": item["favorites"],
                    "tags":           item.get("tag", ""),
                    # 搜索API自带分区名，比详情API的 tname 更可靠
                    "category":       item.get("typename", ""),
                })
                known_bvids.add(bvid)
            page += 1

    # 若第1档未凑齐1200，启用第2档
    if len(new_videos) < 1200:
        print(f"当前 {len(new_videos)} 条，不足1200，放宽分区限制")
        for keyword, order, zone_id, label in FALLBACK_CONFIGS:
            page = 1
            print(f"搜索: {label}")
            while True:
                await random_delay()
                result = await search.search_by_type(
                    keyword=keyword,
                    search_type=SearchObjectType.VIDEO,
                    order_type=order,
                    page=page,
                )
                items = result.get("result", [])
                if not items:
                    break
                for item in items:
                    bvid = item["bvid"]
                    if bvid in known_bvids:
                        continue
                    new_videos.append({...})  # 同上
                    known_bvids.add(bvid)
                page += 1
```

**关键点**：
- 每组搜索配置翻页到返回空才停止，不做数量截断
- `video_zone_type=30` 限定 VOCALOID·UTAU 分区，排除 MMD、游戏等无关结果
- 第1档搜完后若不足1200条，启用第2档（不限分区）作为兜底
- 即使第2档也不够，爬虫正常结束，有多少写多少

### 阶段2：逐条补全详情 + 多P展开

```python
from bilibili_api import video

async def detail_phase():
    records = []
    for v_data in new_videos:
        try:
            await random_delay()
            v = video.Video(bvid=v_data["bvid"])
            info = await v.get_info()

            videos_count = info["videos"]

            if videos_count <= 1:
                # === 单P：一条记录 ===
                records.append(build_record(v_data, info, page=1,
                                             play_estimated=False))
            else:
                # === 多P：筛选IA分P，均分播放量 ===
                pages = info.get("pages", [])
                ia_pages = [p for p in pages
                            if re.search(r'(?<![a-zA-Z])IA(?![a-zA-Z])',
                                         p.get("part", ""))]
                if not ia_pages:
                    continue  # 无IA分P，跳过

                per_page_plays = info["stat"]["view"] // videos_count
                for p in ia_pages:
                    records.append(build_record(
                        v_data, info,
                        page=p["page"],
                        page_title=p.get("part", ""),
                        play_count=per_page_plays,
                        play_estimated=True,
                    ))
        except Exception as e:
            log_error(v_data["bvid"], e)
    return records
```

### 阶段3：写入CSV

```python
CSV_COLUMNS = [
    "aid", "bvid", "page", "mid", "title", "author",
    "publish_date", "duration_sec", "play_count", "danmaku_count",
    "comment_count", "favorite_count", "coin_count", "share_count",
    "like_count", "tags", "category", "url", "copyright",
    "play_estimated",
]

df = pd.DataFrame(records, columns=CSV_COLUMNS)
file_exists = os.path.exists("data/ia_music_data.csv")
df.to_csv("data/ia_music_data.csv", mode="a",
          header=not file_exists, index=False, encoding="utf-8-sig")
```

### 去重策略

- 单P视频：`bvid` 为主键
- 多P视频：`(bvid, page)` 为复合键
- 爬取前加载已有 CSV 的全部 `(bvid, page)` 到 set，命中则跳过

## 完整字段赋值一览

```python
{
    "aid":            item["aid"],                                        # 搜索
    "bvid":           item["bvid"],                                       # 搜索
    "mid":            item["mid"],                                        # 搜索
    "title":          re.sub(r'<[^>]+>', '', item["title"]),              # 搜索
    "author":         item["author"],                                     # 搜索
    "publish_date":   datetime.fromtimestamp(item["pubdate"]),             # 搜索
    "duration_sec":   info["duration"],                                   # 详情(int)
    "play_count":     item["play"],                                       # 搜索
    "danmaku_count":  item["video_review"],                               # 搜索
    "comment_count":  item["review"],                                     # 搜索
    "favorite_count": item["favorites"],                                 # 搜索
    "coin_count":     info["stat"]["coin"],                              # 详情
    "share_count":    info["stat"]["share"],                             # 详情
    "like_count":     info["stat"]["like"],                              # 详情
    "tags":           item.get("tag", ""),                                # 搜索
    "category":       info["tname"],                                      # 详情
    "url":            f"https://www.bilibili.com/video/{item['bvid']}",   # 构造
    "copyright":      info["copyright"],                                  # 详情(辅助)
}
```

## 爬取参数

| 参数 | 值 |
|------|-----|
| 目标总量 | ≥ 1200 条（清洗后保证 ≥1000） |
| **第1档搜索** | VOCALOID·UTAU 分区限定 (tid=30)，3关键词 × 2排序 = 6组配置 |
| **第2档搜索** | 不限分区（仅在第1档不足1200时启用），1关键词 × 2排序 = 2组配置 |
| 搜索策略 | 每组翻页直到返回空，不截断 |
| 排序方式 | `OrderVideo.CLICK` + `OrderVideo.PUBDATE` |
| 请求间隔 | **1~3s 随机**（搜索和详情统一） |
| 重试 | 失败最多重试3次，间隔递增 |
| 412处理 | `sleep(60)` 后重试 |
| 编码 | UTF-8-SIG（含日文标题、特殊字符，BOM兼容Excel） |

## 搜索配置详解

| 档位 | 关键词 | 排序 | 分区 | 意图 |
|------|--------|------|------|------|
| 1 | IA | CLICK | VOCALOID·UTAU | IA标签下的最热门作品 |
| 1 | IA | PUBDATE | VOCALOID·UTAU | IA标签下的最新作品 |
| 1 | IA オリジナル曲 | CLICK | VOCALOID·UTAU | IA原创曲热门 |
| 1 | IA オリジナル曲 | PUBDATE | VOCALOID·UTAU | IA原创曲最新 |
| 1 | IA 翻唱 | CLICK | VOCALOID·UTAU | IA翻唱热门 |
| 1 | IA 翻唱 | PUBDATE | VOCALOID·UTAU | IA翻唱最新 |
| 2 | IA | CLICK | 不限 | 兜底：全网IA热门 |
| 2 | IA | PUBDATE | 不限 | 兜底：全网IA最新 |

## 爬取时间估算

每组搜索配置 B站 最多返回约1000条（≈20页），第1档6组 → 最多约120页搜索调用。

| 阶段 | API调用次数 | 单次间隔 | 总耗时 |
|------|-----------|---------|--------|
| 第1档搜索（6组, VOCALOID限定） | ~120次 | 1~3s随机 | ~4min |
| 第1档去重后 bvid | ~800-1500 个 | - | - |
| 第2档搜索（仅不足时） | ~40次 | 1~3s随机 | ~1min |
| 详情补全 | 等于去重后 bvid 数 | 1~3s随机 | ~30-50min |
| **合计** | - | - | **35-60min** |

> 第1档搜完去重后通常已够1200条，第2档大概率不会触发。详情阶段占绝对大头。

## 异常处理

| 异常情况 | 处理方式 |
|---------|---------|
| `search_by_type()` 返回空 result | 正常终止该关键词的分页循环 |
| `get_info()` 网络超时 | 递增等待后重试，超3次则跳过该条（字段留空，清洗时填0） |
| **412 Precondition Failed**（来自官方FAQ） | 请求太快导致 IP 暂时被封。解决方案：① 降低并发/增大间隔 ② 设置代理 `request_settings.set_proxy("http://proxy:port")` |
| `stat` 某字段缺失 | `info["stat"].get("coin", 0)` 安全取值 |
| `tname` 为空 | 通过 `info["tid"]`（分区ID）映射成分区名作为兜底 |
| title 无 `【】` 标签 | 标题解析返回空字符串，不阻断流程 |
| `copyright` 字段不存在 | 旧版 API 可能无此字段，填 -1 表示未知 |